# Train U-Net landuse segmentation (fond / parking / industriel / friche)

Trains on the tiles produced by `fetch_landuse_dataset.ipynb`. Resumes the attempt halted
in `docs/roadmap_segmentation.md` (read it first), applying its priority fix #1: a
**class-weighted loss** (Dice + weighted CrossEntropy) to counter the extreme pixel
imbalance that made the first attempt's IoU collapse to ~0 on parking/friche. Fix #2
(multiple AOIs instead of one) is handled upstream by the fetch notebook.

**Run `fetch_landuse_dataset.ipynb` first** — this notebook expects tiles in `data/landuse/{images,masks}`.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import torch
from torch.utils.data import random_split

from helpers.dataloaders import build_dataloaders
from helpers.mask_rasterize import CLASS_NAMES, colorize_mask
from helpers.segmentation_dataset import SegmentationTileDataset, compute_class_pixel_weights
from models.unet_builder import build_unet
from training.pipeline_configs import UNET_CONFIG
from training.seg_engine import SegEngine

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Classes ({UNET_CONFIG.num_classes}): {CLASS_NAMES}")

### 1/ Load the tile dataset

In [ ]:
DATA_DIR = Path.cwd().parent / "data" / "landuse"

full_dataset = SegmentationTileDataset(DATA_DIR / "images", DATA_DIR / "masks")
print(f"{len(full_dataset)} tiles")

if len(full_dataset) == 0:
    raise RuntimeError("No tiles found -- run fetch_landuse_dataset.ipynb first.")

### 2/ Split into train / val / test

`build_dataloaders` is the same generic splitter used for EuroSAT (`helpers/dataloaders.py`) -- it doesn't care about the item type, so it works unchanged here.

In [ ]:
BATCH_SIZE = 8
SEED_NUM = 42
train_loader, val_loader, test_loader = build_dataloaders(full_dataset, BATCH_SIZE, 0.7, 0.15, SEED_NUM)

# build_dataloaders only returns DataLoaders; compute_class_pixel_weights below needs the
# raw train Subset, so re-split with the same seed/fractions to get it too.
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
generator = torch.Generator().manual_seed(SEED_NUM)
train_subset, val_subset, test_subset = random_split(full_dataset, [train_size, val_size, test_size], generator=generator)

print(f"train={len(train_subset)} val={len(val_subset)} test={len(test_subset)}")

### 3/ Class weights (fix #1 from the roadmap)

Inverse pixel-frequency weights computed on the **train split only**. Fond will end up
near-1, parking/friche should end up with the largest weights since they're by far the
rarest classes (see docs/roadmap_segmentation.md §5 for the baseline numbers).

In [ ]:
class_weights = compute_class_pixel_weights(train_subset, num_classes=UNET_CONFIG.num_classes)
for name, w in zip(CLASS_NAMES, class_weights.tolist()):
    print(f"  {name}: {w:.3f}")

### 4/ Build the U-Net and the training engine

In [ ]:
model = build_unet(num_classes=UNET_CONFIG.num_classes, device=device)
optimizer = torch.optim.Adam(model.parameters(), lr=UNET_CONFIG.lr)
engine = SegEngine(device, optimizer, num_classes=UNET_CONFIG.num_classes, class_weights=class_weights)
engine.display_info()

### 5/ Train

In [ ]:
EPOCHS = UNET_CONFIG.epochs

history = engine.train_model(model, train_loader, val_loader, EPOCHS, class_names=CLASS_NAMES)

### 6/ Loss and per-class IoU curves

Per-class IoU is what actually matters here -- a rising mean IoU could still hide parking/friche stuck near 0, exactly what happened in the first attempt.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss (Dice + weighted CE)")
axes[0].legend()

iou_per_class = list(zip(*history["val_iou_per_class"]))  # -> one series per class
for name, series in zip(CLASS_NAMES, iou_per_class):
    axes[1].plot(series, label=name)
axes[1].plot(history["val_mean_iou"], label="mean", linestyle="--", color="black")
axes[1].set_title("Validation IoU per class")
axes[1].legend()
plt.tight_layout()
plt.show()

### 7/ Test set evaluation

In [ ]:
test_loss, test_iou = engine.eval_epoch(model, test_loader)
print(f"Test loss: {test_loss:.4f}\n")
print(f"{'Classe':<25}{'IoU':>8}")
for name, iou in zip(CLASS_NAMES, test_iou.tolist()):
    print(f"{name:<25}{iou:>8.4f}")

### 8/ Visualize a few predictions

Image | ground-truth mask (colorized) | predicted mask (colorized), side by side, on test tiles.

In [ ]:
import random

from PIL import Image

MASKS_PREVIEW_DIR = DATA_DIR / "masks_preview"

sample_indices = random.sample(range(len(test_subset)), k=min(4, len(test_subset)))

fig, axes = plt.subplots(len(sample_indices), 3, figsize=(9, 3 * len(sample_indices)))
model.eval()
with torch.no_grad():
    for row, local_idx in enumerate(sample_indices):
        dataset_idx = test_subset.indices[local_idx]
        name = full_dataset.filenames[dataset_idx]

        image_tensor, _ = test_subset[local_idx]
        pred = model(image_tensor.unsqueeze(0).to(device)).argmax(1).squeeze(0).cpu().numpy()

        axes[row, 0].imshow(Image.open(DATA_DIR / "images" / name))
        axes[row, 0].set_title(name)
        axes[row, 1].imshow(Image.open(MASKS_PREVIEW_DIR / name))
        axes[row, 1].set_title("ground truth")
        axes[row, 2].imshow(colorize_mask(pred))
        axes[row, 2].set_title("prediction")
        for ax in axes[row]:
            ax.axis("off")
plt.tight_layout()
plt.show()

### 9/ Save the trained weights

In [ ]:
CHECKPOINT_DIR = Path.cwd().parent / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), CHECKPOINT_DIR / "unet_landuse.pth")
print(f"Saved checkpoint to {CHECKPOINT_DIR / 'unet_landuse.pth'}")